In [1]:
using PEPSKit, TensorKit

### Model Parameters ###
N = 3 #Number of sites in unit cell
n_0 = round(Int, ((N-1)/2))+1 #Index of the point at the center of the lattice (1-based indexing).
m2 = 1 #Bare mass (squared)
m0 = 1 #Basis frequency
l = 0.1 #phi^4 coupling strength
Dim = 4 #Truncated local Hilbert space dimension
d = 2 #Number of spatial dimensions
a = 1 #Lattice spacing

### iPEPS Dimensions ###
Dbond = 3
χ = 24

### Field Operators ###
function a_matrix(D)
    A = zeros(D, D)
    for n in 1:D-1
        A[n, n+1] = sqrt(n)
    end
    return A
end

#Field operator matrices
phi_matrix = D -> (a_matrix(D) + a_matrix(D)')/sqrt(2*m0)
phi2_matrix = D -> phi_matrix(D)^2
phi4_matrix = D -> phi_matrix(D)^4
pi_matrix = D -> im*sqrt(m0/2)*(a_matrix(D)' - a_matrix(D))
pi2_matrix = D -> -(m0 / 2) * ((a_matrix(D)' - a_matrix(D)) * (a_matrix(D)' - a_matrix(D)))

#Truncated Hilbert space (ℂ^D)
V = ComplexSpace(Dim)

#Convert matrices to TensorMap
φ = TensorMap(phi_matrix(Dim), V ← V)
φ2 = TensorMap(phi2_matrix(Dim), V ← V)
φ4 = TensorMap(phi4_matrix(Dim), V ← V)
Π = TensorMap(pi_matrix(Dim), V ← V)
Π2 = TensorMap(pi2_matrix(Dim), V ← V)

### phi4 Hamiltonian ###
function phi4_model(lattice::InfiniteSquare; m2=m2, l=l, a=a, d=d)
    h_site = a^d * ((1/2 * Π2) + (1/a^2 * φ2) + (1/a^2 * φ2) + (1/2 * m2 * φ2) + (l/24 * φ4)) #Onsite Hamiltonian
    h_bond = -1/a^2 * a^d * φ ⊗ φ #Nearest-neighbor interaction Hamiltonian
    spaces = fill(V, (lattice.Nrows, lattice.Ncols))

    return LocalOperator(
        spaces,
        (neighbor => h_bond for neighbor in nearest_neighbours(lattice))...,
        ([idx] => h_site for idx in vertices(lattice))...,
    )
end

H = phi4_model(InfiniteSquare(N, N))

LocalOperator{Any, ComplexSpace}(ComplexSpace[ℂ^4 ℂ^4 ℂ^4; ℂ^4 ℂ^4 ℂ^4; ℂ^4 ℂ^4 ℂ^4], Dict{Vector{CartesianIndex{2}}, Any}([CartesianIndex(3, 3)] => TensorMap{Float64, ComplexSpace, 1, 1, Vector{Float64}}([1.5031249999999998, 0.0, 1.4230523971379268, 0.0, 0.0, 4.515625, 0.0, 2.4647990536755726, 1.4230523971379268, 0.0, 7.528124999999999, 0.0, 0.0, 2.4647990536755726, 0.0, 4.515624999999999], ℂ^4 ← ℂ^4), [CartesianIndex(2, 1), CartesianIndex(3, 1)] => TensorMap{Float64, ComplexSpace, 2, 2, Vector{Float64}}([0.0, 0.0, 0.0, 0.0, 0.0, -0.4999999999999999, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, -1.4999999999999998, 0.0, 0.0, 0.0, 0.0, 0.0], (ℂ^4 ⊗ ℂ^4) ← (ℂ^4 ⊗ ℂ^4)), [CartesianIndex(1, 2)] => TensorMap{Float64, ComplexSpace, 1, 1, Vector{Float64}}([1.5031249999999998, 0.0, 1.4230523971379268, 0.0, 0.0, 4.515625, 0.0, 2.4647990536755726, 1.4230523971379268, 0.0, 7.528124999999999, 0.0, 0.0, 2.4647990536755726, 0.0, 4.515624999999999], ℂ^4 ← ℂ^4), [CartesianIndex(3, 3), CartesianIndex(4,

In [2]:
### Ground state optimization parameters ###
boundary_alg = (; tol = 1e-8, trunc = (; alg = :FixedSpaceTruncation));
optimizer_alg = (; alg = :LBFGS, tol = 1e-4, maxiter = 200, lbfgs_memory = 16);
reuse_env = true
verbosity = 3;

### Initialize PEPS and CTMRG environment
peps₀ = InfinitePEPS(randn, ComplexF64, ℂ^Dim, ℂ^Dbond; unitcell=(N, N))
env_random = CTMRGEnv(randn, ComplexF64, peps₀, ℂ^χ);
env₀, info_ctmrg = leading_boundary(env_random, peps₀; boundary_alg...);

[ Info: CTMRG init:	obj = +1.594115004355e+09 +1.504101302382e+09im	err = 1.0000e+00
[ Info: CTMRG conv 22:	obj = +1.121565977198e+14 -6.796875000000e-01im	err = 5.4290462716e-09	time = 1.05 min


In [3]:
### Find the ground state ∣Ω⟩ ###
peps_gs, env_gs, E, info_opt = fixedpoint(H, peps₀, env₀; boundary_alg, optimizer_alg, reuse_env, verbosity);

[ Info: LBFGS: initializing with f = 4.128201781354e+01, ‖∇f‖ = 1.9747e+01
[ Info: LBFGS: iter    1, Δt  6.82 m: f = 1.685339683206e+01, ‖∇f‖ = 1.0821e+01, α = 2.18e+02, m = 0, nfg = 7
[ Info: LBFGS: iter    2, Δt  1.30 m: f = 1.339537725099e+01, ‖∇f‖ = 1.7090e+01, α = 1.00e+00, m = 1, nfg = 1
[ Info: LBFGS: iter    3, Δt 58.43 s: f = 1.089172817929e+01, ‖∇f‖ = 6.2397e+00, α = 1.00e+00, m = 2, nfg = 1
[ Info: LBFGS: iter    4, Δt 55.42 s: f = 1.029019113758e+01, ‖∇f‖ = 3.8105e+00, α = 1.00e+00, m = 3, nfg = 1
[ Info: LBFGS: iter    5, Δt 53.69 s: f = 9.743317100938e+00, ‖∇f‖ = 3.9489e+00, α = 1.00e+00, m = 4, nfg = 1
[ Info: LBFGS: iter    6, Δt 47.76 s: f = 9.018699040943e+00, ‖∇f‖ = 3.1257e+00, α = 1.00e+00, m = 5, nfg = 1
[ Info: LBFGS: iter    7, Δt  1.52 m: f = 8.908531105598e+00, ‖∇f‖ = 2.5205e+00, α = 4.91e-01, m = 6, nfg = 2
[ Info: LBFGS: iter    8, Δt 43.86 s: f = 8.818906676318e+00, ‖∇f‖ = 9.6891e-01, α = 1.00e+00, m = 7, nfg = 1
[ Info: LBFGS: iter    9, Δt 43.57 s: f = 8.7

In [8]:
using JLD2
save_object("VacStates/PEPS,N=$N,m2=$m2,l=$l,a=$a,dim=$Dim,D=$Dbond,chi=$χ.jld2", peps_gs)
save_object("VacStates/env,N=$N,m2=$m2,l=$l,a=$a,dim=$Dim,D=$Dbond,chi=$χ.jld2", env_gs)

In [ ]:
#testPEPS = load_object("VacStates/PEPS,N=$N,m2=$m2,l=$l,a=$a,dim=$Dim,D=$Dbond,chi=$χ.jld2")

InfinitePEPS{TensorMap{ComplexF64, ComplexSpace, 1, 4, Vector{ComplexF64}}}(TensorMap{ComplexF64, ComplexSpace, 1, 4, Vector{ComplexF64}}[TensorMap{ComplexF64, ComplexSpace, 1, 4, Vector{ComplexF64}}(ComplexF64[0.08865881604225836 - 0.06901114270023236im, -0.005284804606681369 + 0.03940009904440531im, -0.033895462243386916 + 0.04577590894526153im, 0.0009701629927038923 - 0.011884115921409153im, -0.003309598384804825 + 0.08057841258200163im, -0.001651654497239157 + 0.011649518082817374im, 0.010508442135778402 - 0.02121397130453849im, 0.002869127040324005 - 0.02015963593763005im, -0.08340873726220085 + 0.03570988659212107im, 0.002175895734105736 - 0.028098541597490038im  …  0.042441242001507645 + 0.0029680683618137395im, -0.07241617034926177 - 0.030745810154506278im, 0.02002974039168132 - 0.11121880116930154im, -0.023896914584096252 + 0.014401487636614238im, -0.005447515871993451 + 0.019958054367415344im, 0.002031818410694385 - 0.005385242565911144im, 0.08038177552026318 - 0.064905341979